In [2]:
import pandas as pd

df = pd.read_csv("../data/processed/repositories.csv", keep_default_na=False)

In [3]:
df.head(1)

,Full Name,Repository Name,Description,Topics,Primary Language,Stars Count,Forks Count,Updated At,Domain,combined_text
0,robertpeteuil/multi-cloud-control,multi-cloud-control,Multi cloud control of VM Instances across AWS...,"alibaba-cloud, alibaba-cloud-cli, alibabacloud...",Python,35,10,2024-03-14 22:40:50+00:00,Cloud Computing,multi cloud control of vm instances across aws...


**Vectorizer**

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=10000,
    ngram_range=(1,2),
    min_df= 2
)

X = vectorizer.fit_transform(df['combined_text'])

In [5]:
X.shape

(4253, 10000)

In [6]:
vectorizer.get_feature_names_out()[:50]

array(['10', '10 windows', '100', '100 days', '100 free', '100 languages',
       '1000', '100daysofcode', '1011', '1024', '10x', '10x faster', '11',
       '118', '12', '12 weeks', '13', '150', '16', '180', '1wire', '1x',
       '20', '20 components', '2017', '2018', '2019', '2020', '2020 3d',
       '2021', '2022', '2022 ai', '2023', '2024', '2024 agent',
       '2024 computervision', '2025', '2025 ai', '2026', '2026 awesome',
       '2026 codinginterviewquestions', '21', '23', '24', '24 lessons',
       '24ghz', '25', '26', '2d', '2d 2dframework'], dtype=object)

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
similarities = cosine_similarity(X[0], X)

In [9]:
similarities.shape

(1, 4253)

In [10]:
similarities[0][:10]

array([1.        , 0.06922728, 0.15419042, 0.        , 0.11194199,
       0.21037124, 0.07860243, 0.09429201, 0.11954501, 0.09140685])

In [11]:
similarity_score = list(enumerate(similarities[0]))

In [12]:
similarity_score = sorted(
    similarity_score,
    key=lambda x: x[1],
    reverse=True
)

In [13]:
similarity_score[:10]

[(0, np.float64(1.0)),
 (39, np.float64(0.45839454644555594)),
 (867, np.float64(0.40230539279047683)),
 (14, np.float64(0.3901551360865684)),
 (3518, np.float64(0.3886444712717679)),
 (320, np.float64(0.37707244924321653)),
 (123, np.float64(0.317334282608472)),
 (1694, np.float64(0.28492855693763414)),
 (2054, np.float64(0.276503329695475)),
 (126, np.float64(0.2763770083565774))]

In [14]:
indices = [i[0] for i in similarity_score[1:11]]

df.iloc[indices][["Repository Name", "Description", "Primary Language", "Stars Count"]]

,Repository Name,Description,Primary Language,Stars Count
39,alibabacloud-console-design,阿里云管平台研发解决方案,TypeScript,83
867,Cloud-Product-Mapping,"All major services between AWS, Azure, and GCP...",,907
14,cloud-cheat-sheets,My handmade cheat-sheets for different AWS ser...,,97
3518,Cloud-Free-Tier-Comparison,Comparing the free tier offers of the major cl...,,6862
320,skyplane,🔥 Blazing fast bulk data transfers between any...,Python,1211
123,terraform-provider-iterative,☁️ Terraform plugin for machine learning workl...,Go,295
1694,docker-android,Android in docker solution with noVNC supporte...,Python,14471
2054,cb-tumblebug,Cloud-Barista Multi-Cloud Infra Management Fra...,Go,80
126,AzureR,Family of packages for interacting with Azure ...,,197
81,90DaysOfGoogleCloudPlatform,,,83


**Recommendation Function**

In [15]:
# Recommendation Function

def recommend(full_name, n=10):

    repo_indices = df.index[df["Full Name"] == full_name].tolist()

    if not repo_indices:
        return "Repository not found"

    idx = repo_indices[0]

    repo_vector = X[idx]

    similarities = cosine_similarity(repo_vector, X)[0] # 1-d array

    similarity_scores = list(enumerate(similarities))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Remove the repo itself
    similarity_scores = similarity_scores[1:n+1]

    indices = [i[0] for i in similarity_scores]
    scores = [i[1] for i in similarity_scores]

    recommendations = df.iloc[indices][[
        "Full Name",
        "Repository Name",
        "Description",
        "Domain",
        "Primary Language",
        "Stars Count",
        "Forks Count",
        "Updated At"
    ]].copy()

    recommendations["Similarity"] = scores

    return recommendations


In [16]:
df["Full Name"].sample(5,random_state=42).tolist()

['alsiam/web-projects',
 'tink-crypto/tink',
 'joewalnes/websocketd',
 'microsoft/Data-Science-For-Beginners',
 'latent-to/bittensor']

In [17]:
recommend('alsiam/web-projects')

,Full Name,Repository Name,Description,Domain,Primary Language,Stars Count,Forks Count,Updated At,Similarity
443,unknwon/building-web-applications-in-go,building-web-applications-in-go,Go 语言 Web 应用开发系列教程，从新手到双手残废,Web Development,Go,566,40,2026-04-04 21:32:01+00:00,0.329385
969,keshavgbpecdelhi/Web-Development,Web-Development,Here you will find different web development m...,Web Development,JavaScript,1522,489,2026-04-09 02:32:49+00:00,0.328118
2429,adobe/brackets,brackets,"An open source code editor for the web, writte...",JavaScript,JavaScript,33079,7509,2026-04-10 07:26:48+00:00,0.326850
1084,ItzAshOffcl/awesome-webdev-resources,awesome-webdev-resources,A curated list of useful websites and resource...,Web Development,,189,15,2026-04-09 08:01:24+00:00,0.320607
737,sajalagrawal/LifestyleStore,LifestyleStore,An Online Shopping website[Built for learning ...,Web Development,PHP,138,83,2026-04-08 07:10:11+00:00,0.314446
84,swlkr/ryde,ryde,"ryde is a single person, single file web frame...",Web Development,Rust,326,8,2026-02-20 00:05:43+00:00,0.289200
3334,mdn/content,content,The official source for MDN Web Docs content. ...,Web Development,Markdown,10658,23171,2026-04-10 09:14:00+00:00,0.277773
86,pakyow/pakyow,pakyow,Design-First Web Framework,Web Development,Ruby,816,63,2026-02-20 11:20:20+00:00,0.276254
956,jstrieb/urlpages,urlpages,Create and view web pages stored entirely in t...,Web Development,JavaScript,1384,132,2026-04-09 01:50:20+00:00,0.273522
1240,pure-css/pure,pure,"A set of small, responsive CSS modules that yo...",JavaScript,JavaScript,23752,2424,2026-04-09 10:41:33+00:00,0.265863


**EVALUATION**

In [18]:
eval_repos = df["Full Name"].sample(20, random_state=42 ).tolist()

evaluation = []

for repo in eval_repos:
    recommendations = recommend(repo, 5)

    for _, row in recommendations.iterrows():
        evaluation.append({
            "Query Repo": repo,
            "Recommended Repo": row["Repository Name"],
            "Similarity": row["Similarity"],
            "Relevant": None
        })

eval_df = pd.DataFrame(evaluation)


In [19]:
eval_df

,Query Repo,Recommended Repo,Similarity,Relevant
0,alsiam/web-projects,building-web-applications-in-go,0.329385,None
1,alsiam/web-projects,Web-Development,0.328118,None
2,alsiam/web-projects,brackets,0.326850,None
3,alsiam/web-projects,awesome-webdev-resources,0.320607,None
4,alsiam/web-projects,LifestyleStore,0.314446,None
...,...,...,...,...
95,Zhefan-Xu/NavRL,RVO2,0.278374,None
96,Zhefan-Xu/NavRL,IsaacLab,0.273715,None
97,Zhefan-Xu/NavRL,skrl,0.266839,None
98,Zhefan-Xu/NavRL,RoboTwin,0.264126,None


In [20]:
eval_df[
    ["Query Repo", "Recommended Repo", "Similarity", "Relevant"]
]

,Query Repo,Recommended Repo,Similarity,Relevant
0,alsiam/web-projects,building-web-applications-in-go,0.329385,None
1,alsiam/web-projects,Web-Development,0.328118,None
2,alsiam/web-projects,brackets,0.326850,None
3,alsiam/web-projects,awesome-webdev-resources,0.320607,None
4,alsiam/web-projects,LifestyleStore,0.314446,None
...,...,...,...,...
95,Zhefan-Xu/NavRL,RVO2,0.278374,None
96,Zhefan-Xu/NavRL,IsaacLab,0.273715,None
97,Zhefan-Xu/NavRL,skrl,0.266839,None
98,Zhefan-Xu/NavRL,RoboTwin,0.264126,None


In [21]:
eval_df.to_csv("../data/evaluation.csv",index=False)

**Final Evaluation**

In [22]:
evaluation = pd.read_csv("../data/evaluation_audited.csv")

In [23]:
precision_at_5 = evaluation.groupby("Query Repo")["Relevant"].mean()

precision_at_5

Query Repo
Zhefan-Xu/NavRL                            1.0
alsiam/web-projects                        1.0
cisagov/ScubaGear                          1.0
exosphere-project/exosphere                0.6
go-ego/gse                                 0.8
hermit-os/hermit-playground                0.6
houbb/sensitive-word                       0.6
joewalnes/websocketd                       1.0
killop/anything_about_game                 1.0
latent-to/bittensor                        0.8
microsoft/Data-Science-For-Beginners       1.0
milanaryal/web-development-resources       1.0
projectlombok/lombok                       0.0
psviderski/uncloud                         1.0
rust-lang/mdBook                           0.2
snorkel-team/snorkel                       1.0
spatie/laravel-backup                      0.4
tink-crypto/tink                           0.2
vcmi/vcmi                                  0.6
voxsim/awesome-software-engineer-topics    1.0
Name: Relevant, dtype: float64

In [24]:
mean_p_at_5 = precision_at_5.mean()

print(f"Mean Precision@5: {mean_p_at_5:.2%}")

Mean Precision@5: 74.00%


**Saving the model**

In [25]:
import joblib

joblib.dump(vectorizer, "../models/tfidf_vectorizer.pkl")
joblib.dump(X, "../models/tfidf_matrix.pkl")

['../models/tfidf_matrix.pkl']

## Hybrid Model

**Popularity Score**

In [ ]:
df[['Stars Count', 'Forks Count', 'Updated At']].head()

,Stars Count,Forks Count,Updated At
0,35,10,2024-03-14 22:40:50+00:00
1,70,12,2024-05-09 07:20:49+00:00
2,86,11,2024-08-15 19:57:55+00:00
3,213,106,2024-11-01 00:54:36+00:00
4,36,11,2024-11-21 11:43:08+00:00


In [ ]:
df[['Stars Count', 'Forks Count', 'Updated At']].describe()

,Stars Count,Forks Count
count,4253.000000,4253.000000
mean,14919.591818,2353.453092
std,24579.199772,5172.234224
min,35.000000,0.000000
25%,1873.000000,243.000000
50%,6260.000000,790.000000
75%,19584.000000,2405.000000
max,420610.000000,106780.000000


In [37]:
df['Stars Count'].gt(100000).sum()

np.int64(53)

In [38]:
df['Stars Count'].gt(50000).sum()

np.int64(238)

In [39]:
df['Stars Count'].gt(20000).sum()

np.int64(1042)

In [40]:
df["Stars Count"].quantile([0.5, 0.75, 0.90, 0.95, 0.99])

0.50      6260.00
0.75     19584.00
0.90     36213.00
0.95     53005.40
0.99    109601.92
Name: Stars Count, dtype: float64

*The Stars Count data is right skewed*

**Transforming Data**

In [41]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

df['log_stars'] = np.log1p(df['Stars Count'])
df['log_forks'] = np.log1p(df['Forks Count'])

scaler = MinMaxScaler()

df[["stars_score", "forks_score"]] = scaler.fit_transform(df[["log_stars","log_forks"]])

# Popularity score
df["popularity"] = (
    0.5 * df["stars_score"]
    + 0.5 * df["forks_score"]
)

In [42]:
df[[
    "Full Name",
    "Stars Count",
    "Forks Count",
    "stars_score",
    "forks_score",
    "popularity"]].sort_values("popularity",ascending=False).head(10)

,Full Name,Stars Count,Forks Count,stars_score,forks_score,popularity
4224,jwasham/coding-interview-university,340694,81910,0.977501,0.977100,0.977301
3780,EbookFoundation/free-programming-books,385281,66098,0.990633,0.958576,0.974605
4076,public-apis/public-apis,420610,45769,1.000000,0.926834,0.963417
3831,donnemartin/system-design-primer,342104,55267,0.977942,0.943120,0.960531
4173,ultraworkers/claw-code,180261,106780,0.909534,1.000000,0.954767
4123,tensorflow/tensorflow,194622,75263,0.917718,0.969791,0.943755
4124,facebook/react,244480,50912,0.942070,0.936031,0.939051
4004,TheAlgorithms/Python,219470,50316,0.930547,0.935014,0.932781
4057,vinta/awesome-python,291616,27629,0.960894,0.883243,0.922068
4228,Significant-Gravitas/AutoGPT,183290,46226,0.911313,0.927692,0.919503


## EXPERIMENTS ##

**Checking Popularity score influence on recommendation**

In [43]:
# Recommender
def hybrid_recommend(full_name, n=5, lambda_=0.1):

    repo_indices = df.index[
        df["Full Name"] == full_name
    ].tolist()

    if not repo_indices:
        return "Repository not found"

    idx = repo_indices[0]

    repo_vector = X[idx]

    similarities = cosine_similarity(
        repo_vector, X
    )[0]

    final_scores = (
        (1 - lambda_) * similarities
        + lambda_ * df["popularity"].values
    )

    # Don't recommend the query repository itself
    final_scores[idx] = -1

    top_indices = np.argsort(
        final_scores
    )[::-1][:n]

    recommendations = df.iloc[top_indices][[
        "Full Name",
        "Repository Name",
        "Description",
        "Domain",
        "Primary Language",
        "Stars Count",
        "Forks Count",
        "Updated At"
    ]].copy()

    recommendations["Similarity"] = similarities[top_indices]
    recommendations["Popularity"] = df.iloc[top_indices]["popularity"].values
    recommendations["Final Score"] = final_scores[top_indices]

    return recommendations

**Checking Domain feature's influence**

In [27]:
df["text_without_domain"] = (
    df["Description"] + " " +
    df["Topics"]
)

df["text_with_domain"] = (
    df["Description"] + " " +
    df["Topics"] + " " +
    df["Domain"]
)

In [28]:
vectorizer_a = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2)
)

X_without_domain = vectorizer_a.fit_transform(
    df["text_without_domain"]
)

In [29]:
vectorizer_b = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2)
)

X_with_domain = vectorizer_b.fit_transform(
    df["text_with_domain"]
)

In [30]:
X_without_domain.shape


(4253, 10000)

In [31]:
X_with_domain.shape

(4253, 10000)

In [32]:
from sklearn.metrics.pairwise import cosine_similarity

bitnet_idx = df.index[
    df["Repository Name"] == "BitNet"
][0]

sim_without = cosine_similarity(
    X_without_domain[bitnet_idx],
    X_without_domain
)[0]

sim_with = cosine_similarity(
    X_with_domain[bitnet_idx],
    X_with_domain
)[0]

In [33]:
top_without = sim_without.argsort()[::-1][1:6]
top_with = sim_with.argsort()[::-1][1:6]

In [34]:
print("WITHOUT DOMAIN")
print(df.iloc[top_without][["Repository Name", "Domain"]])

print("\nWITH DOMAIN")
print(df.iloc[top_with][["Repository Name", "Domain"]])

WITHOUT DOMAIN
          Repository Name                                 Domain
2739                llama                                 Python
2792  Reverse-Engineering                          Cybersecurity
1374      agibot_x1_infer                               Robotics
1685               litgpt  Deep Learning Artificial Intelligence
1360             AutoGPTQ            Natural Language Processing

WITH DOMAIN
       Repository Name  Domain
3919  awesome-llm-apps  Python
4002            crewAI  Python
2739             llama  Python
3634             12306  Python
2211       shadowsocks  Python


*Domain provides useful categorical information, but it shouldn't be the main source of similarity.*